# Jupiter flux review — phase2 QA Zarr

Interactive summary of Stokes I flux toward **Jupiter** in pipeline QA Zarr stores
(`pipelineQA-phase2-I-NoTaper-Robust-0-*.zarr`).

For the selected observation day:

1. **Ephemeris** — Astropy `get_body("jupiter", …)` at the **first** dataset time step.
2. **Dynamic spectrum** — flux vs LST × frequency at that fixed RA/Dec. Methods:
   `dynamic_spectrum` (tracked centre pixel), `patch_fit` (shifted 2D Gaussian),
   or `patch_max` (maximum in the tracked patch). Click a cell for patch-fit
   diagnostics when using `patch_fit`.
3. **Sky view** — click a cell in the dynamic spectrum to show that time/frequency
   slice in `astrowidget.SkyWidget`, centered on Jupiter at observation start.

Paths and Zarr naming follow `pipeline_qa_check_phase2.ipynb`.

Flux extraction uses Dask-backed batched I/O and (for `patch_fit`) multiprocess
Gaussian fitting — see **Parallelism notes** at the bottom.

Launch with: `pixi run jupyter lab`

In [1]:
from dataclasses import replace
from pathlib import Path

from ovro_lwa_portal.viz.pipeline_qa import PipelineQAConfig

# Edit before running if your staging paths differ.
ZARR_ROOT = Path("/fast/claw")
I_QA_ZARR_STEM = "pipelineQA-phase2-I-NoTaper-Robust-0"

QA_CONFIG = replace(
    PipelineQAConfig.phase2_default(),
    zarr_root=ZARR_ROOT,
    i_qa_zarr_stem=I_QA_ZARR_STEM,
)

# Flux extraction: "dynamic_spectrum" (tracked pixel) or "patch_fit" (Gaussian peak).
FLUX_METHOD = "patch_fit"
# Patch half-width = ceil(scale * max beam FWHM in pixels) at each time/frequency.
PATCH_FIT_SCALE = 3.0
PATCH_FIT_MAX_REDUCED_CHI_SQUARED = 200.0

# Optional distributed Dask (see the Dask cell below).
# Default off: radport uses threaded I/O + local process pools for patch work.
USE_DASK_CLIENT = False
DASK_WORKERS = 6
DASK_THREADS_PER_WORKER = 1
DASK_MEMORY_LIMIT = "16GiB"
DASK_PROCESSES = True  # True for CPU-bound patch_fit; False for I/O-only debugging

In [2]:
import warnings

warnings.filterwarnings("ignore")

import math
import time
from collections.abc import Callable

import ipywidgets as widgets
import numpy as np
import ovro_lwa_portal as ovro
import panel as pn
import param
import xarray as xr
import astropy.units as u
from astropy.coordinates import SkyCoord, get_body
from astropy.time import Time
from astropy.utils.iers import conf as iers_conf
from astrowidget import SkyWidget
from bokeh.events import Tap
from bokeh.models import ColumnDataSource, FixedTicker, HoverTool, LinearColorMapper
from bokeh.palettes import Inferno256
from bokeh.plotting import figure

from ovro_lwa_portal.accessor import format_radec_sexagesimal
from ovro_lwa_portal.viz.pipeline_qa import PipelineQAConfig
from ovro_lwa_portal.viz.pipeline_qa_app import (
    ACTIVITY_LOG_HEIGHT_PX,
    ScrollLog,
    _format_activity_log_html,
    _patch_astrowidget_get_wcs,
    _push_panel_layout,
    _schedule_ipython_main,
)

JUPITER_SKY_FOV_DEG = 10.0

_patch_astrowidget_get_wcs()
pn.extension("bokeh", sizing_mode="stretch_width")

In [3]:
def list_phase2_i_qa_zarrs(config: PipelineQAConfig) -> list[Path]:
    """Return sorted Stokes I phase2 QA Zarr paths under ``config.zarr_root``."""
    pattern = f"{config.i_qa_zarr_stem}-*.zarr"
    return sorted(config.zarr_root.glob(pattern))


def zarr_path_to_day(path: Path, *, stem: str) -> str:
    """Parse ``YYYY-MM-DD`` from a QA Zarr directory name."""
    day_tag = path.name.removeprefix(f"{stem}-").removesuffix(".zarr")
    if len(day_tag) != 8 or not day_tag.isdigit():
        return path.name
    return f"{day_tag[:4]}-{day_tag[4:6]}-{day_tag[6:8]}"


def jupiter_at_observation_start(ds: xr.Dataset) -> SkyCoord:
    """Jupiter FK5 coordinates at the first time sample in the dataset."""
    mjd = float(np.asarray(ds.coords["time"].values, dtype=np.float64)[0])
    orig = iers_conf.auto_download
    try:
        iers_conf.auto_download = False
        t0 = Time(mjd, format="mjd", scale="utc")
        return get_body("jupiter", t0)
    finally:
        iers_conf.auto_download = orig


def lst_hours_for_dataset(ds: xr.Dataset) -> np.ndarray:
    """Mean local sidereal time (hours) for each dataset time sample."""
    from astropy.coordinates import EarthLocation

    observatory = EarthLocation.of_site("ovro")
    mjd = np.asarray(ds.coords["time"].values, dtype=np.float64)
    orig = iers_conf.auto_download
    try:
        iers_conf.auto_download = False
        times = Time(mjd, format="mjd", scale="utc")
        lst_deg = np.asarray(times.sidereal_time("mean", longitude=observatory.lon).deg)
    finally:
        iers_conf.auto_download = orig
    return np.mod(lst_deg / 15.0, 24.0)


_PROGRESS_STAGE_LABELS = {
    "track": "Pixel track",
    "extract": "Pixel I/O",
    "reduce": "Statistics",
    "fit": "Patch fit",
}


def jupiter_flux_map(
    ds: xr.Dataset,
    jupiter: SkyCoord,
    *,
    method: str = "dynamic_spectrum",
    patch_fit_scale: float = 3.0,
    patch_fit_max_reduced_chi_squared: float = 3.0,
    progress_callback: Callable[[str, int, int, str], None] | None = None,
) -> tuple[xr.DataArray, object | None]:
    """Flux map and optional :class:`~ovro_lwa_portal.accessor.PatchFitResult`."""
    ra = float(jupiter.ra.deg)
    dec = float(jupiter.dec.deg)
    if method == "dynamic_spectrum":
        flux = ds.radport.dynamic_spectrum(
            ra=ra, dec=dec, progress_callback=progress_callback
        )
        flux.attrs["flux_method"] = "dynamic_spectrum"
        return flux, None
    if method == "patch_max":
        stat = ds.radport.patch_statistic(
            ra=ra,
            dec=dec,
            statistic="max",
            scale=patch_fit_scale,
            progress_callback=progress_callback,
        )
        flux = stat.stat_map
        flux.name = "flux"
        flux.attrs["flux_method"] = "patch_max"
        return flux, None
    if method == "patch_fit":
        fit = ds.radport.patch_fit(
            ra=ra,
            dec=dec,
            scale=patch_fit_scale,
            max_reduced_chi_squared=patch_fit_max_reduced_chi_squared,
            allow_position_offset=True,
            progress_callback=progress_callback,
        )
        flux = fit.peak_map
        flux.name = "flux"
        flux.attrs["flux_method"] = "patch_fit"
        return flux, fit
    msg = (
        f"Unknown flux method {method!r}; expected "
        "'dynamic_spectrum', 'patch_fit', or 'patch_max'"
    )
    raise ValueError(msg)


def _format_scalar_hover(value: float, *, fmt: str = ".3g") -> str:
    if np.isfinite(value):
        return format(float(value), fmt)
    return "n/a"


def _patch_fit_hover_columns(fit: object) -> dict[str, list[str]]:
    """Pre-formatted Bokeh hover fields for a full patch-fit result."""
    chi2 = np.asarray(fit.reduced_chi_squared_map.values, dtype=np.float64)
    peak = np.asarray(fit.peak_map.values, dtype=np.float64)
    x_off = np.asarray(fit.x_offset_map.values, dtype=np.float64)
    y_off = np.asarray(fit.y_offset_map.values, dtype=np.float64)
    pmax = np.asarray(fit.patch_max_map.values, dtype=np.float64)
    accepted = np.asarray(fit.fit_accepted_map.values, dtype=bool)
    peak_ra, peak_dec = fit.peak_radec_maps()
    ra = np.asarray(peak_ra.values, dtype=np.float64)
    dec = np.asarray(peak_dec.values, dtype=np.float64)

    def _row(arr: np.ndarray) -> list[str]:
        return [_format_scalar_hover(float(v)) for v in arr.ravel()]

    peak_ra_display: list[str] = []
    peak_dec_display: list[str] = []
    for r, d in zip(ra.ravel(), dec.ravel(), strict=True):
        ra_s, dec_s = format_radec_sexagesimal(float(r), float(d))
        peak_ra_display.append(ra_s)
        peak_dec_display.append(dec_s)

    return {
        "chi2_display": _row(chi2),
        "peak_ra_display": peak_ra_display,
        "peak_dec_display": peak_dec_display,
        "offset_display": [
            (
                f"({x:.2f}, {y:.2f})"
                if np.isfinite(x) and np.isfinite(y)
                else "n/a"
            )
            for x, y in zip(x_off.ravel(), y_off.ravel(), strict=True)
        ],
        "fit_accepted_display": ["yes" if a else "no" for a in accepted.ravel()],
        "patch_max_display": _row(pmax),
        "peak_flux_display": [
            f"{v:.3g} (masked)" if not np.isfinite(v) else f"{v:.3g}"
            for v in peak.ravel()
        ],
    }


def _format_patch_fit_diagnostics(fit: object, time_idx: int, freq_idx: int) -> str:
    """Markdown summary for one patch-fit cell."""
    diag = fit.cell_diagnostics(time_idx=time_idx, frequency_idx=freq_idx)
    accepted = "yes" if diag["fit_accepted"] else "no (χ² above cut)"
    peak = diag["peak"]
    peak_s = f"{peak:.3g}" if np.isfinite(peak) else "n/a (masked)"
    ra_s = str(diag["peak_ra"])
    dec_s = str(diag["peak_dec"])
    return (
        f"**patch_fit** t={time_idx} f={freq_idx}: accepted={accepted}, "
        f"χ²_red={diag['reduced_chi_squared']:.3g}, peak={peak_s} Jy, "
        f"peak RA/Dec=({ra_s}, {dec_s}), "
        f"offset=({diag['x_offset_pixels']:.2f}, {diag['y_offset_pixels']:.2f}) px, "
        f"centre={diag['center_flux']:.3g} Jy, patch_max={diag['patch_max']:.3g} Jy"
    )


def _heatmap_index_from_coord(coord: float, n: int) -> int:
    if n <= 0:
        return 0
    return int(np.clip(int(np.floor(float(coord))), 0, n - 1))


def _format_lst_hour_label(lst_hour: float) -> str:
    hour = int(round(float(lst_hour))) % 24
    return f"{hour:02d}h"


def _color_mapper(values: np.ndarray) -> LinearColorMapper:
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return LinearColorMapper(palette=Inferno256, low=0.0, high=1.0)
    lo, hi = np.percentile(finite, [2, 98])
    if hi <= lo:
        hi = lo + 1.0
    return LinearColorMapper(
        palette=Inferno256,
        low=float(lo),
        high=float(hi),
        # Opaque gray so failed patch_fit cells are not confused with dark colormap lows.
        nan_color="#9e9e9e",
    )


def _format_flux_hover(values: np.ndarray) -> list[str]:
    """One hover label per (time, freq) cell; non-finite flux reads as n/a."""
    return [f"{float(v):.3g}" if np.isfinite(v) else "n/a" for v in values.ravel()]

In [4]:
class JupiterFluxReview(param.Parameterized):
    """Dynamic spectrum toward Jupiter with linked SkyWidget."""

    select_zarr = param.Selector(default=None, objects=[], doc="Phase2 QA Zarr store.")
    flux_method = param.Selector(
        default="dynamic_spectrum",
        objects=["dynamic_spectrum", "patch_fit", "patch_max"],
        doc="Flux: tracked pixel, Gaussian patch fit, or patch maximum.",
    )
    loading = param.Boolean(default=False)
    status = param.String(default="Select a QA Zarr store.")
    log_text = param.String(default="")

    def __init__(
        self,
        config: PipelineQAConfig,
        *,
        patch_fit_scale: float = PATCH_FIT_SCALE,
        patch_fit_max_reduced_chi_squared: float = PATCH_FIT_MAX_REDUCED_CHI_SQUARED,
        **params,
    ) -> None:
        self._config = config
        self._patch_fit_scale = patch_fit_scale
        self._patch_fit_max_reduced_chi_squared = patch_fit_max_reduced_chi_squared
        self._scroll_log = ScrollLog()
        self._zarr_paths: dict[str, Path] = {}
        self._dataset: xr.Dataset | None = None
        self._dynspec: xr.DataArray | None = None
        self._patch_fit_result: object | None = None
        self._jupiter: SkyCoord | None = None
        self._lst_hours: np.ndarray | None = None
        self._freq_mhz: np.ndarray | None = None
        self._sky_widget: SkyWidget | None = None
        self._time_idx = 0
        self._freq_idx = 0

        zarr_paths = list_phase2_i_qa_zarrs(config)
        if not zarr_paths:
            labels: list[str] = []
        else:
            labels = []
            for path in zarr_paths:
                day = zarr_path_to_day(path, stem=config.i_qa_zarr_stem)
                label = f"{day} ({path.name})"
                labels.append(label)
                self._zarr_paths[label] = path

        default = labels[-1] if labels else None
        super().__init__(select_zarr=default, **params)
        self.param.select_zarr.objects = labels

        self._heatmap_pane = pn.pane.Bokeh(height=420, sizing_mode="stretch_width")
        self._sky_container = widgets.VBox(
            children=[widgets.HTML("<i>Sky view loads after selecting a Zarr store.</i>")],
            layout=widgets.Layout(width="100%", min_height="620px"),
        )
        self._sky_pane = pn.pane.IPyWidget(self._sky_container, height=620, sizing_mode="stretch_width")
        self._status_pane = pn.pane.Markdown("")
        self._log_pane = pn.pane.HTML(
            _format_activity_log_html(""),
            sizing_mode="stretch_width",
            height=ACTIVITY_LOG_HEIGHT_PX,
        )
        self._selector = pn.widgets.Select.from_param(
            self.param.select_zarr,
            name="Phase2 QA Zarr (Stokes I)",
            width=520,
        )
        self._flux_method_selector = pn.widgets.Select.from_param(
            self.param.flux_method,
            name="Flux method",
            width=200,
        )
        self._spinner = pn.indicators.LoadingSpinner(value=False, size=24, name="")
        self._layout = pn.Column(
            pn.Row(
                self._selector,
                self._flux_method_selector,
                self._spinner,
                margin=(0, 0, 8, 0),
            ),
            pn.Column(
                pn.pane.Markdown("**Activity log**"),
                self._log_pane,
                sizing_mode="stretch_width",
            ),
            self._status_pane,
            self._heatmap_pane,
            self._sky_pane,
            sizing_mode="stretch_width",
            max_width=1048,
        )
        self.param.watch(self._on_select_zarr, "select_zarr")
        self.param.watch(self._on_flux_method_change, "flux_method")
        if labels:
            self._log(f"Found {len(labels)} phase2 QA Zarr store(s).")
        elif not labels:
            self._log("No phase2 QA Zarr stores found under the configured root.")
        if default is not None:
            self._load_selected_zarr()
        self._log_dask_dashboard()

    @property
    def panel(self) -> pn.Column:
        return self._layout

    def _on_select_zarr(self, *_events) -> None:
        self._load_selected_zarr()

    def _on_flux_method_change(self, *_events) -> None:
        if self.select_zarr is not None and not self.loading:
            self._load_selected_zarr()

    def _flux_method_label(self) -> str:
        if self.flux_method == "patch_fit":
            return "patch_fit (shifted Gaussian)"
        if self.flux_method == "patch_max":
            return "patch_max (patch maximum)"
        return "dynamic_spectrum (tracked pixel)"

    def _set_status(self, text: str) -> None:
        self.status = text
        self._status_pane.object = text

    @param.depends("log_text", watch=True)
    def _sync_log_pane(self) -> None:
        self._log_pane.object = _format_activity_log_html(self.log_text)

    def _sync_log(self) -> None:
        self.log_text = self._scroll_log.text

    def _log(self, message: str) -> None:
        self._scroll_log.append(message)
        self._sync_log()
        _push_panel_layout(self._layout, self._log_pane)

    def _log_dask_dashboard(self) -> None:
        try:
            from dask.distributed import get_client

            client = get_client()
            self._log(f"Dask dashboard: {client.dashboard_link}")
        except Exception:
            pass

    def _flux_progress_callback(self) -> Callable[[str, int, int, str], None]:
        last_key: dict[str, tuple[int, str]] = {}

        def _callback(stage: str, current: int, total: int, message: str) -> None:
            if stage in ("extract", "track") and total > 1:
                key = (current, message)
                if current not in (0, total) and last_key.get(stage) == key:
                    return
                last_key[stage] = key
            label = _PROGRESS_STAGE_LABELS.get(stage, stage)
            if "in progress" in message or (stage == "extract" and current < total):
                text = f"{label}: {message}"
            else:
                pct = int(round(100.0 * int(current) / int(total))) if total else 0
                text = f"{label}: {message} ({current}/{total}, {pct}%)"

            def _push() -> None:
                self._log(text)

            _schedule_ipython_main(_push)

        return _callback

    def _load_selected_zarr(self) -> None:
        label = self.select_zarr
        if label is None:
            self._set_status("No phase2 QA Zarr stores found under the configured root.")
            return
        path = self._zarr_paths[label]
        self.loading = True
        self._spinner.value = True
        self._log(f"Loading {path.name}…")
        self._set_status("Loading…")
        _push_panel_layout(self._layout, self._status_pane, self._spinner, self._log_pane)

        def _work() -> None:
            t0 = time.perf_counter()
            try:
                _schedule_ipython_main(
                    lambda: self._log(f"Opening Stokes I Zarr at {path}…")
                )
                ds = ovro.open_dataset(path, chunks="auto").chunk({"l": 512, "m": 512})
                _schedule_ipython_main(
                    lambda: self._log(
                        f"Opened Zarr ({int(ds.sizes['time'])} times, "
                        f"{int(ds.sizes['frequency'])} frequencies, "
                        f"{int(ds.sizes['l'])}×{int(ds.sizes['m'])} pixels)."
                    )
                )
                _schedule_ipython_main(
                    lambda: self._log("Computing Jupiter ephemeris at observation start…")
                )
                jupiter = jupiter_at_observation_start(ds)
                method_label = self._flux_method_label()
                _schedule_ipython_main(
                    lambda: self._log(
                        f"Extracting flux map via {method_label} toward Jupiter "
                        f"(RA={float(jupiter.ra.deg):.3f}°, Dec={float(jupiter.dec.deg):.3f}°)…"
                    )
                )
                dynspec, patch_fit = jupiter_flux_map(
                    ds,
                    jupiter,
                    method=self.flux_method,
                    patch_fit_scale=self._patch_fit_scale,
                    patch_fit_max_reduced_chi_squared=self._patch_fit_max_reduced_chi_squared,
                    progress_callback=self._flux_progress_callback(),
                )
                elapsed = time.perf_counter() - t0
                _schedule_ipython_main(
                    lambda: self._log(
                        f"Flux extraction finished in {elapsed:.1f} s "
                        f"({method_label})."
                    )
                )
                lst_hours = lst_hours_for_dataset(ds)
                freq_mhz = np.asarray(ds.coords["frequency"].values, dtype=np.float64) / 1e6
            except Exception as exc:
                _schedule_ipython_main(
                    lambda: self._finish_load(None, None, None, None, None, None, exc)
                )
                return
            _schedule_ipython_main(
                lambda: self._finish_load(
                    ds, dynspec, jupiter, lst_hours, freq_mhz, patch_fit, None
                )
            )

        import threading

        threading.Thread(target=_work, daemon=True).start()

    def _finish_load(
        self,
        ds: xr.Dataset | None,
        dynspec: xr.DataArray | None,
        jupiter: SkyCoord | None,
        lst_hours: np.ndarray | None,
        freq_mhz: np.ndarray | None,
        patch_fit_result: object | None,
        error: BaseException | None,
    ) -> None:
        self.loading = False
        self._spinner.value = False
        if error is not None:
            self._log(f"ERROR: {error}")
            self._set_status(f"**Load failed:** {error}")
            _push_panel_layout(self._layout, self._status_pane, self._spinner, self._log_pane)
            return

        assert ds is not None and dynspec is not None
        assert jupiter is not None
        assert lst_hours is not None and freq_mhz is not None

        self._dataset = ds
        self._dynspec = dynspec
        self._patch_fit_result = patch_fit_result
        self._jupiter = jupiter
        self._lst_hours = lst_hours
        self._freq_mhz = freq_mhz
        if patch_fit_result is not None:
            n_accepted = int(patch_fit_result.fit_accepted_map.sum().values)
            n_cells = int(patch_fit_result.fit_accepted_map.size)
            self._log(
                f"patch_fit quality: {n_accepted}/{n_cells} cells accepted "
                f"(χ²_red ≤ {self._patch_fit_max_reduced_chi_squared:g})"
            )

        jupiter_ra = jupiter.ra.to_string(unit=u.hour, precision=1)
        jupiter_dec = jupiter.dec.to_string(unit=u.deg, precision=1)
        self._set_status(
            f"**Jupiter at observation start:** RA={jupiter_ra}, Dec={jupiter_dec} · "
            f"{int(ds.sizes['time'])} times × {int(ds.sizes['frequency'])} frequencies · "
            f"Flux: {self._flux_method_label()} · "
            "Click the dynamic spectrum to center the sky view on Jupiter."
        )

        self._mount_sky_widget(ds)
        self._time_idx, self._freq_idx = self._default_slice(dynspec.values)
        self._heatmap_pane.object = self._build_dynspec_figure(dynspec.values)
        self._update_sky(self._time_idx, self._freq_idx)
        self._log(
            f"Ready — Jupiter dynamic spectrum and sky view loaded for "
            f"{self._zarr_paths[self.select_zarr].name}."
        )
        _push_panel_layout(
            self._layout, self._status_pane, self._heatmap_pane, self._sky_pane, self._log_pane
        )

    def _default_slice(self, values: np.ndarray) -> tuple[int, int]:
        finite = np.argwhere(np.isfinite(values))
        if finite.size:
            t_idx, f_idx = finite[len(finite) // 2]
            return int(t_idx), int(f_idx)
        return 0, 0

    def _mount_sky_widget(self, ds: xr.Dataset) -> None:
        widget = SkyWidget()
        widget.colormap = "inferno"
        widget.background_survey = ""
        widget.invert_horizontal_pan = True
        max_size = max(256, int(ds.sizes["l"]) // 2)
        widget.set_dataset(ds, max_size=max_size)
        self._sky_widget = widget
        self._sky_container.children = [widget]

    def _update_sky(self, time_idx: int, freq_idx: int) -> None:
        widget = self._sky_widget
        jupiter = self._jupiter
        if widget is None or jupiter is None:
            return
        widget.update_slice(
            time_idx=int(time_idx),
            freq_idx=int(freq_idx),
            center=jupiter,
            fov=JUPITER_SKY_FOV_DEG * u.deg,
            percentile_low=2,
            percentile_high=98,
        )
        send_state = getattr(widget, "send_state", None)
        if callable(send_state):
            send_state()

    def _on_heatmap_tap(self, time_idx: int, freq_idx: int) -> None:
        self._time_idx = time_idx
        self._freq_idx = freq_idx
        if self._jupiter is None:
            return
        ra = self._jupiter.ra.to_string(unit=u.hour, precision=1)
        dec = self._jupiter.dec.to_string(unit=u.deg, precision=1)
        lst = _format_lst_hour_label(float(self._lst_hours[time_idx]))
        freq = float(self._freq_mhz[freq_idx])
        status = (
            f"**Selected slice:** LST {lst}, {freq:.1f} MHz · "
            f"Jupiter (t₀) RA={ra}, Dec={dec}"
        )
        if self.flux_method == "patch_fit" and self._patch_fit_result is not None:
            diag_md = _format_patch_fit_diagnostics(
                self._patch_fit_result, time_idx, freq_idx
            )
            status = f"{status}\n\n{diag_md}"
            self._log(diag_md.replace("**patch_fit** ", "patch_fit "))
        self._set_status(status)
        self._log(
            f"Sky view updated — time {time_idx}, freq {freq_idx} "
            f"({freq:.1f} MHz), FOV {JUPITER_SKY_FOV_DEG:.0f}°."
        )
        self._update_sky(time_idx, freq_idx)
        _push_panel_layout(self._layout, self._status_pane, self._sky_pane, self._log_pane)

    def _build_dynspec_figure(self, values: np.ndarray):
        n_times, n_freqs = values.shape
        mapper = _color_mapper(values.astype(np.float64, copy=False))

        if self.flux_method == "patch_fit":
            plot_title = (
                "Gaussian-fit peak flux toward Jupiter — hover for χ², peak RA/Dec, offsets"
            )
            flux_tooltip = ("Peak flux (Jy/beam)", "@peak_flux_display")
        elif self.flux_method == "patch_max":
            plot_title = "Patch-max flux toward Jupiter (RA/Dec at observation start)"
            flux_tooltip = ("Patch max (Jy/beam)", "@flux_display")
        else:
            plot_title = "Dynamic spectrum toward Jupiter (RA/Dec at observation start)"
            flux_tooltip = ("Flux (Jy/beam)", "@flux_display")
        plot = figure(
            width=1000,
            height=400,
            title=plot_title,
            x_range=(0, n_times),
            y_range=(0, n_freqs),
            tools="pan,wheel_zoom,reset,tap",
            active_drag="pan",
            active_tap="tap",
        )
        # Bokeh image rows = y (frequency), cols = x (time); values is (time, freq).
        plot.image(
            image=[values.T.astype(np.float64, copy=False)],
            x=0,
            y=0,
            dw=n_times,
            dh=n_freqs,
            color_mapper=mapper,
        )

        time_idx, freq_idx = np.meshgrid(
            np.arange(n_times, dtype=int),
            np.arange(n_freqs, dtype=int),
            indexing="ij",
        )
        flat_time = time_idx.ravel()
        flat_freq = freq_idx.ravel()
        hover_data: dict[str, object] = {
            "x": flat_time + 0.5,
            "y": flat_freq + 0.5,
            "time_idx": flat_time,
            "freq_idx": flat_freq,
            "lst_hour": [
                _format_lst_hour_label(float(h))
                for h in self._lst_hours[flat_time]
            ],
            "freq_mhz": self._freq_mhz[flat_freq],
            "flux_display": _format_flux_hover(values),
        }
        tooltips: list[tuple[str, str]] = [
            ("LST hour", "@lst_hour"),
            ("Freq (MHz)", "@freq_mhz{0.1}"),
            ("Time idx", "@time_idx"),
            ("Freq idx", "@freq_idx"),
            flux_tooltip,
        ]
        if self.flux_method == "patch_fit" and self._patch_fit_result is not None:
            hover_data.update(_patch_fit_hover_columns(self._patch_fit_result))
            tooltips.extend(
                [
                    ("Patch max (Jy)", "@patch_max_display"),
                    ("χ²_red", "@chi2_display"),
                    ("Fit accepted", "@fit_accepted_display"),
                    ("Peak RA", "@peak_ra_display"),
                    ("Peak Dec", "@peak_dec_display"),
                    ("Offset (l,m px)", "@offset_display"),
                ]
            )
        hover_src = ColumnDataSource(data=hover_data)
        hover_renderer = plot.rect(
            x="x",
            y="y",
            width=1,
            height=1,
            source=hover_src,
            fill_alpha=0,
            line_alpha=0,
        )
        plot.add_tools(
            HoverTool(
                renderers=[hover_renderer],
                tooltips=tooltips,
            )
        )

        def _axis_ticks(n: int, values: np.ndarray, fmt: Callable) -> tuple[list[float], dict[float, str]]:
            step = 1 if n <= 24 else int(np.ceil(n / 24))
            indices = range(0, n, step)
            ticks = [i + 0.5 for i in indices]
            labels = {tick: fmt(values[i]) for tick, i in zip(ticks, indices, strict=True)}
            return ticks, labels

        x_ticks, x_labels = _axis_ticks(n_times, self._lst_hours, _format_lst_hour_label)
        y_ticks, y_labels = _axis_ticks(n_freqs, self._freq_mhz, lambda v: f"{float(v):.1f}")
        plot.xaxis.ticker = FixedTicker(ticks=x_ticks)
        plot.yaxis.ticker = FixedTicker(ticks=y_ticks)
        plot.xaxis.major_label_overrides = x_labels
        plot.yaxis.major_label_overrides = y_labels
        plot.xaxis.axis_label = "LST hour"
        plot.yaxis.axis_label = "Frequency (MHz)"
        plot.xaxis.major_label_orientation = math.pi / 4

        def _on_tap(event: Tap) -> None:
            if event.x is None or event.y is None:
                return
            t_idx = _heatmap_index_from_coord(event.x, n_times)
            f_idx = _heatmap_index_from_coord(event.y, n_freqs)
            _schedule_ipython_main(lambda: self._on_heatmap_tap(t_idx, f_idx))

        plot.on_event(Tap, _on_tap)
        return plot

In [5]:
if USE_DASK_CLIENT:
    from dask.distributed import Client, get_client

    try:
        dask_client = get_client()
    except ValueError:
        dask_client = Client(
            n_workers=DASK_WORKERS,
            threads_per_worker=DASK_THREADS_PER_WORKER,
            processes=DASK_PROCESSES,
            memory_limit=DASK_MEMORY_LIMIT,
        )
    print(dask_client)
    print(f"Dashboard: {dask_client.dashboard_link}")
else:
    print(
        "Dask Client disabled — radport uses threaded Zarr I/O and a local "
        "process pool for patch_fit / patch_max reductions."
    )

Dask Client disabled — radport uses threaded Zarr I/O and a local process pool for patch_fit / patch_max reductions.


In [6]:
review = JupiterFluxReview(
    QA_CONFIG,
    flux_method=FLUX_METHOD,
    patch_fit_scale=PATCH_FIT_SCALE,
    patch_fit_max_reduced_chi_squared=PATCH_FIT_MAX_REDUCED_CHI_SQUARED,
)
review.panel

Column(max_width=1048, sizing_mode='stretch_width')
    [0] Row(margin=(0, 0, 8, 0), sizing_mode='stretch_width')
        [0] Select(description='Phase2 QA Zarr store.', name='Phase2 QA Zarr (..., width=520)
        [1] Select(description='Flux: tracked pixel, ..., name='Flux method', options=OrderedDict({'dynamic_spec...]), value='patch_fit', width=200)
        [2] LoadingSpinner(size=24)
    [1] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
        [1] HTML(str, height=150, sizing_mode='stretch_width')
    [2] Markdown(str, sizing_mode='stretch_width')
    [3] Bokeh(None, height=420, sizing_mode='stretch_width')
    [4] IPyWidget(VBox, height=620, sizing_mode='stretch_width')

## Parallelism notes

Jupiter QA Zarr stores use **per-time WCS** (`wcs_header_str(time)`), so RA/Dec
tracking maps a **different pixel each time step** before reading flux.

| Stage | `dynamic_spectrum` | `patch_fit` / `patch_max` |
| --- | --- | --- |
| **track** | Bulk header parse + in-process `world2pix` (one WCS object, CRVAL updated per time) | Same |
| **extract** | One vectorized Zarr read (`isel` with per-time `l`/`m` index arrays) — **threaded** dask | Same for point reads; patches still batched |
| **reduce / fit** | — | Per-time statistics or Gaussian fit — **process pool** locally, or distributed workers when a `Client` is active |

**Why one core before:** the default `dask.compute()` scheduler is threaded; scipy
Gaussian fitting holds the GIL, so `patch_fit` stayed on a single CPU. The
accessor now uses `scheduler="threads"` for I/O and `scheduler="processes"` for
CPU-bound patch work when no distributed `Client` is running.

**Recommendations**

- **`dynamic_spectrum`** — leave `USE_DASK_CLIENT = False`; threaded batched reads
  are usually fastest and avoid shipping tiny tasks to workers.
- **`patch_fit`** — default (no Client) is fine after the accessor change; set
  `USE_DASK_CLIENT = True` with `DASK_PROCESSES = True` only if you want the
  dashboard or to tune worker count / memory on very long days.
- Watch the activity log for phased progress (`Pixel track`, `Pixel I/O`, `Patch fit`).